# QMC.jl for Lebesgue Integration

This notebook follows the structure of QMCPy's `lebesgue_integration.ipynb` as closely as the Julia API allows.

QMC.jl currently constructs `Lebesgue` directly from a discrete distribution rather than wrapping another true measure, so the volume factor is written explicitly in the integrands below when matching the Python notebook's numerics.


In [1]:
using QMC
using Printf


In [2]:
abs_tol = 0.01
dim = 1
a = 0.0
b = 2.0
true_value = 8.0 / 3.0


2.6666666666666665

In [3]:
tm = Lebesgue(Halton(dim; seed=7, replications=64); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.3f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = 2.667


false

In [4]:
tm = Uniform(Halton(dim; seed=7, replications=64); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> (b - a) .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Uniform measure:  y = %.3f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Uniform measure:  y = 2.667


false

In [5]:
abs_tol = 0.001
dim = 2
a = [1.0, 2.0]
b = [2.0, 4.0]
true_value = ((a[1]^3 - b[1]^3) * (a[2] - b[2]) + (a[1] - b[1]) * (a[2]^3 - b[2]^3)) / 3
@printf("Answer = %.5f\n", true_value)


Answer = 23.33333


In [6]:
tm = Lebesgue(DigitalNetB2(dim; seed=7, randomize="LMS_DS"); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.5f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = 23.33333


false

In [7]:
tm = Uniform(DigitalNetB2(dim; seed=17, randomize="LMS_DS"); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> prod(b .- a) .* vec(sum(x .^ 2, dims=2)))
solution = integrate(CubMCCLT(integrand; abs_tol=abs_tol)).solution
@printf("Uniform measure:  y = %.5f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Uniform measure:  y = 23.33333


false

In [8]:
abs_tol = 1e-4
dim = 1
a = 3.0
b = 5.0
true_value = -0.87961


-0.87961

In [9]:
tm = Lebesgue(Lattice(dim; randomize=true, seed=7); lower_bound=a, upper_bound=b)
integrand = CustomFun(tm, x -> tm.volume .* vec(sin.(x) ./ log.(x)))
solution = integrate(CubQMCLatticeG(integrand; abs_tol=abs_tol)).solution
@printf("Lebesgue measure: y = %.3f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Lebesgue measure: y = -0.880


false

In [10]:
abs_tol = 0.1
dim = 2
true_value = π


π = 3.1415926535897...

In [11]:
# QMC.jl does not currently implement Lebesgue(Gaussian(...)) directly.
# Instead, integrate with respect to a Gaussian measure and include the
# Lebesgue-to-Gaussian weight explicitly in the integrand.
tm = Gaussian(Lattice(dim; randomize=true, seed=7); mean=0.0, covariance=1.0)
integrand = CustomFun(tm, x -> vec((2π)^(dim / 2) .* prod(exp.(-x .^ 2 ./ 2), dims=2)))
solution = integrate(CubQMCLatticeG(integrand; abs_tol=abs_tol)).solution
@printf("Gaussian measure with Lebesgue weight: y = %.3f\n", solution)
abs(solution - true_value) > abs_tol && error("Not within error tolerance")


Gaussian measure with Lebesgue weight: y = 3.142


false